# Advanced Pandas Data Analysis — Practice Notebook (English)

This notebook uses the same `sales_data.csv` dataset, but goes into **harder, more advanced pandas concepts**.

Every code cell has **line-by-line comments** explaining exactly what each line does.

Dataset columns:
- `OrderID`, `Date`, `Region`, `Category`, `Product`, `Quantity`, `UnitPrice`, `TotalSales`, `CustomerAge`, `Rating`

Each section has: explanation → fully commented example code → a harder practice exercise for you to try → solution.


## 0. Setup — Load and Clean the Data First

In [ ]:
import pandas as pd            # main library for tabular data analysis
import numpy as np             # for numerical operations and NaN handling
import matplotlib.pyplot as plt  # for plotting charts

# Load the CSV file into a DataFrame
df = pd.read_csv('sales_data.csv')          # reads the csv file from disk into a DataFrame object

# Remove duplicate rows (there are some intentionally added in the dataset)
df = df.drop_duplicates()                   # returns a new DataFrame with duplicate rows removed

# Fill missing CustomerAge values with the column mean
mean_age = df['CustomerAge'].mean()         # calculate the average age, ignoring NaNs automatically
df['CustomerAge'] = df['CustomerAge'].fillna(mean_age)   # replace NaN values with the mean age

# Convert the Date column from text to actual datetime objects
df['Date'] = pd.to_datetime(df['Date'])     # now pandas understands this column as real dates, not strings

df.head()                                   # display the first 5 rows to confirm everything loaded correctly


## 1. GroupBy with Multiple Aggregations (`agg()`)

This is a harder concept: instead of one aggregation (like `.sum()`), we can apply **different aggregation functions to different columns at once** using `.agg()`.

In [ ]:
# Group the data by Region AND Category at the same time
grouped = df.groupby(['Region', 'Category'])          # creates groups for every unique Region+Category combination

# Apply DIFFERENT aggregation functions to different columns in one call
summary = grouped.agg(
    total_sales=('TotalSales', 'sum'),          # sum of TotalSales, renamed to 'total_sales'
    avg_price=('UnitPrice', 'mean'),            # mean of UnitPrice, renamed to 'avg_price'
    order_count=('OrderID', 'count'),           # count of orders, renamed to 'order_count'
    max_quantity=('Quantity', 'max')            # maximum quantity sold in a single order
)

summary = summary.reset_index()   # converts the Region/Category group-index back into normal columns

summary.head(10)                  # show the first 10 rows of this summarized table


### Practice 1.1 (harder)
Using `.agg()`, group the data by `Category` only, and calculate:
- the **median** of `TotalSales`
- the **standard deviation** of `UnitPrice`
- the number of **unique Regions** each category was sold in (hint: use `'Region': 'nunique'`)

Store the result in a variable called `category_stats`.


In [ ]:
# Write your code here


In [ ]:
# Solution
category_stats = df.groupby('Category').agg(
    median_sales=('TotalSales', 'median'),      # middle value of TotalSales per category
    price_std=('UnitPrice', 'std'),             # spread/variation of prices per category
    unique_regions=('Region', 'nunique')        # how many different regions sold this category
)

category_stats   # display the result


## 2. Pivot Tables

A pivot table reshapes data: it turns unique values of one column into new columns, which is very useful for comparison tables.

In [ ]:
# Build a pivot table: rows = Region, columns = Category, values = sum of TotalSales
pivot = pd.pivot_table(
    df,                          # source DataFrame
    values='TotalSales',         # the numbers we want to aggregate
    index='Region',              # becomes the row labels
    columns='Category',          # each unique Category becomes its own column
    aggfunc='sum',                # how to combine multiple rows that match (sum here)
    fill_value=0                 # replace any missing combination with 0 instead of NaN
)

pivot   # display the reshaped table


### Practice 2.1 (harder)
Create a pivot table where:
- rows = `Month` (you'll need to create this column first from `Date`)
- columns = `Region`
- values = `Quantity`
- aggregation = `sum`

Also add a `margins=True` argument so pandas automatically adds a "Total" row and column.


In [ ]:
# Write your code here


In [ ]:
# Solution
df['Month'] = df['Date'].dt.month_name()   # extract the full month name from the Date column

monthly_region_pivot = pd.pivot_table(
    df,
    values='Quantity',
    index='Month',
    columns='Region',
    aggfunc='sum',
    fill_value=0,
    margins=True,          # adds a 'All' row and column showing totals
    margins_name='Total'   # renames the totals row/column to 'Total'
)

monthly_region_pivot


## 3. `apply()` with Custom Functions and `lambda`

`apply()` lets you run your own function on every row or every value — much more flexible than built-in methods.

In [ ]:
# Define a custom function that classifies a sale as Low, Medium, or High value
def classify_sale(amount):          # takes a single number as input
    if amount < 200:                # check if it's below 200
        return 'Low'
    elif amount < 800:              # check if it's between 200 and 800
        return 'Medium'
    else:                           # anything 800 or above
        return 'High'

# Apply this function to every value in the TotalSales column
df['SaleCategory'] = df['TotalSales'].apply(classify_sale)   # creates a new column based on the function's output

df[['TotalSales', 'SaleCategory']].head(10)   # preview the new column next to the original values


### Practice 3.1 (harder)
Write a function `age_group(age)` that returns:
- `'Teen'` if age < 20
- `'Young Adult'` if 20 <= age < 35
- `'Adult'` if 35 <= age < 50
- `'Senior'` otherwise

Apply it to `CustomerAge` to create a new column `AgeGroup`.
Then use `apply()` with a **lambda function** (not a full `def`) to create a column `PricePerUnitRounded` that rounds `UnitPrice` to the nearest 10 (hint: `round(x / 10) * 10`).


In [ ]:
# Write your code here


In [ ]:
# Solution
def age_group(age):                     # custom function taking one age value
    if age < 20:
        return 'Teen'
    elif age < 35:
        return 'Young Adult'
    elif age < 50:
        return 'Adult'
    else:
        return 'Senior'

df['AgeGroup'] = df['CustomerAge'].apply(age_group)     # apply function row by row

df['PricePerUnitRounded'] = df['UnitPrice'].apply(lambda x: round(x / 10) * 10)  # inline lambda function

df[['CustomerAge', 'AgeGroup', 'UnitPrice', 'PricePerUnitRounded']].head(10)


## 4. Merging DataFrames (`merge`)

In real projects, data often comes from multiple tables that need to be joined together — similar to SQL joins.

In [ ]:
# Create a small extra DataFrame with a discount percentage for each Region
discount_data = pd.DataFrame({
    'Region': ['Punjab', 'Sindh', 'KPK', 'Balochistan', 'Islamabad'],   # must match values in df['Region']
    'DiscountPercent': [5, 7, 3, 10, 4]                                  # made-up discount per region
})

# Merge the main dataframe with the discount table, matching rows on 'Region'
df_merged = df.merge(
    discount_data,       # the other table to join
    on='Region',         # the shared column to match rows on
    how='left'           # keep all rows from df, even if no match is found in discount_data
)

# Calculate the discounted sales amount using the newly merged column
df_merged['DiscountedSales'] = df_merged['TotalSales'] * (1 - df_merged['DiscountPercent'] / 100)

df_merged[['Region', 'TotalSales', 'DiscountPercent', 'DiscountedSales']].head(10)


### Practice 4.1 (harder)
Create a small DataFrame `category_tax` with a `TaxPercent` for each `Category` (make up your own values).
Merge it into `df` using `how='left'`, then calculate a new column `FinalPrice` = `TotalSales` + tax amount.


In [ ]:
# Write your code here


In [ ]:
# Solution
category_tax = pd.DataFrame({
    'Category': ['Electronics', 'Clothing', 'Groceries', 'Furniture', 'Toys'],
    'TaxPercent': [17, 12, 5, 15, 10]
})

df_tax = df.merge(category_tax, on='Category', how='left')     # join tax rates onto the main data

df_tax['FinalPrice'] = df_tax['TotalSales'] * (1 + df_tax['TaxPercent'] / 100)  # add tax on top of sales

df_tax[['Category', 'TotalSales', 'TaxPercent', 'FinalPrice']].head(10)


## 5. String Operations (`.str` accessor)

Pandas has a `.str` accessor that lets you run string methods (like `.upper()`, `.contains()`) on an entire column at once.

In [ ]:
# Convert Product names to uppercase
df['Product_Upper'] = df['Product'].str.upper()          # applies .upper() to every value in the column

# Check which products contain the word 'Phone' (case-insensitive)
phone_products = df[df['Product'].str.contains('Phone', case=False)]   # boolean filter based on substring match

# Extract just the first word of each Product name
df['Product_FirstWord'] = df['Product'].str.split(' ').str[0]  # split on space, take the first element of each list

df[['Product', 'Product_Upper', 'Product_FirstWord']].head()


### Practice 5.1 (harder)
Create a new column `OrderNumber` that extracts only the numeric part of `OrderID` (e.g. 'ORD1000' → 1000) as an **integer**.
Hint: use `.str.replace('ORD', '')` then `.astype(int)`.


In [ ]:
# Write your code here


In [ ]:
# Solution
df['OrderNumber'] = df['OrderID'].str.replace('ORD', '', regex=False).astype(int)  # remove prefix, convert to integer

df[['OrderID', 'OrderNumber']].head()


## 6. Time Series: Resampling and Rolling Windows

Since `Date` is a real datetime column, we can do time-based analysis like daily totals, weekly averages, and rolling (moving) averages.

In [ ]:
# Set Date as the index so pandas can do time-based operations
df_ts = df.set_index('Date').sort_index()      # sort by date after setting it as index

# Resample: get total sales PER WEEK (summing all days in each week)
weekly_sales = df_ts['TotalSales'].resample('W').sum()   # 'W' means weekly frequency

# Calculate a 4-week rolling (moving) average to smooth out the trend
rolling_avg = weekly_sales.rolling(window=4).mean()       # average of the current + previous 3 weeks

# Plot both the raw weekly sales and the smoothed rolling average
plt.figure(figsize=(10,4))
weekly_sales.plot(label='Weekly Sales', alpha=0.5)         # thin line for actual weekly totals
rolling_avg.plot(label='4-Week Rolling Average', linewidth=2)  # thicker line for the smoothed trend
plt.legend()
plt.title('Weekly Sales with Rolling Average')
plt.tight_layout()
plt.show()


### Practice 6.1 (harder)
Resample the data by **month** (`'ME'`) instead of week, and calculate the **average** `Rating` per month (not sum).
Then calculate a **2-month rolling average** of that monthly rating and plot both lines.


In [ ]:
# Write your code here


In [ ]:
# Solution
monthly_rating = df_ts['Rating'].resample('ME').mean()          # average rating per calendar month

rolling_rating = monthly_rating.rolling(window=2).mean()        # smooth it with a 2-month rolling average

plt.figure(figsize=(10,4))
monthly_rating.plot(label='Monthly Avg Rating', marker='o')     # actual monthly averages with dot markers
rolling_rating.plot(label='2-Month Rolling Avg', linewidth=2)   # smoothed trend line
plt.legend()
plt.title('Monthly Rating Trend')
plt.tight_layout()
plt.show()


## 7. Working with MultiIndex (Hierarchical Index)

When you group by more than one column, pandas creates a MultiIndex. Knowing how to select data from it is an important advanced skill.

In [ ]:
# Group by Region and Category, creating a MultiIndex Series
multi = df.groupby(['Region', 'Category'])['TotalSales'].sum()   # index has two levels: Region, then Category

# Select ALL rows for one specific Region (first level of the index)
punjab_only = multi.loc['Punjab']            # returns just the Category-level data for Punjab

# Select ONE specific Region+Category combination
specific = multi.loc[('Punjab', 'Electronics')]   # pass a tuple to select an exact combination

# Convert the MultiIndex Series back into a flat DataFrame with normal columns
flat_df = multi.reset_index()                # 'Region' and 'Category' become normal columns again

print(punjab_only)
print("Punjab Electronics total:", specific)
flat_df.head()


### Practice 7.1 (harder)
Create a MultiIndex Series grouped by `['Category', 'Region']` (note the order is reversed) summing `Quantity`.
Then select only the data for `'Electronics'`, and within that, sort it so the region with the highest quantity appears first.


In [ ]:
# Write your code here


In [ ]:
# Solution
multi2 = df.groupby(['Category', 'Region'])['Quantity'].sum()   # first level = Category, second level = Region

electronics_data = multi2.loc['Electronics']         # select only the Electronics section of the index

electronics_sorted = electronics_data.sort_values(ascending=False)   # order regions by highest quantity first

electronics_sorted


## 8. Memory Optimization with Categorical Data

Large datasets can use a lot of memory. Converting text columns with few unique values into the `category` dtype saves memory significantly.

In [ ]:
# Check memory usage BEFORE optimization
print("Before optimization:")
print(df.memory_usage(deep=True).sum(), "bytes")   # deep=True correctly measures memory used by text columns

# Convert repetitive text columns to the 'category' dtype
df_optimized = df.copy()                                    # work on a copy so original df is untouched
df_optimized['Region'] = df_optimized['Region'].astype('category')      # only 5 unique values
df_optimized['Category'] = df_optimized['Category'].astype('category')  # only 5 unique values

# Check memory usage AFTER optimization
print("After optimization:")
print(df_optimized.memory_usage(deep=True).sum(), "bytes")


### Practice 8.1 (harder)
Convert the `Product` column to `category` dtype as well, and print the memory usage difference (before vs after) just for that one column, in kilobytes (KB).


In [ ]:
# Write your code here


In [ ]:
# Solution
before_kb = df['Product'].memory_usage(deep=True) / 1024        # convert bytes to kilobytes

df_optimized['Product'] = df_optimized['Product'].astype('category')  # convert to category dtype

after_kb = df_optimized['Product'].memory_usage(deep=True) / 1024    # measure again after conversion

print(f"Before: {before_kb:.2f} KB")
print(f"After: {after_kb:.2f} KB")
print(f"Saved: {before_kb - after_kb:.2f} KB")


## 9. Final Challenge (Hardest) 🎯

Try to solve this completely on your own, combining everything learned above:

1. Create a pivot table showing the **average Rating** for each `Region` (rows) and `SaleCategory` (columns) — you created `SaleCategory` in Section 3.
2. Using `.agg()`, group by `Category` and find, in a single call: total sales, average customer age, and the number of distinct products sold.
3. Create a rolling 7-day average of daily total sales (resample by day `'D'` first, then apply a rolling window of 7).
4. Write a function using `apply()` that flags an order as `'Suspicious'` if `Quantity > 15` AND `UnitPrice > 400`, otherwise `'Normal'`. Add it as a new column.

Write your solution in the cell below.


In [ ]:
# Write your final challenge solution here

